In [2]:
import numpy as np
import pandas as pd
import os
import gc

!pip install mlflow dagshub -q

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_PASSWORD"] = user_secrets.get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_USERNAME"] = user_secrets.get_secret("DAGSHUB_USERNAME")

import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow")
mlflow.set_experiment("RandomForest_Training")

print("MLflow connected!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 608.6 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 32.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

train_transaction = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_identity = pd.read_csv(f"{DATA_DIR}/train_identity.csv")

train = train_transaction.merge(train_identity, on="TransactionID", how="left")

del train_transaction, train_identity
gc.collect()

print(f"Train shape: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean():.4f}")

Train shape: (590540, 434)
Fraud rate: 0.0350


# EDA
პირველ რიგში, შევხედოთ მონაცემებს და გავარკვიოთ რამდენია კატეგორიული ცვლადი, რამდენია ცარიელი, რომელ ფიჩერებს აქვთ ყველაზე მეტი ცარიელი მნიშვნელობა

In [4]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numerical: {len(num_cols)}, Categorical: {len(cat_cols)}")

missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct}).query("n_missing > 0").sort_values("pct_missing", ascending=False)
print(f"Columns with missing: {len(missing_df)} of {train.shape[1]}")
print(f"Top 10 most missing:")
print(missing_df.head(10))

Numerical: 403, Categorical: 31
Columns with missing: 414 of 434
Top 10 most missing:
       n_missing  pct_missing
id_24     585793        99.20
id_26     585377        99.13
id_25     585408        99.13
id_21     585381        99.13
id_07     585385        99.13
id_08     585385        99.13
id_23     585371        99.12
id_22     585371        99.12
id_27     585371        99.12
dist2     552913        93.63


# Data Separation
მონაცემების 80/20 გაყოფა train და validation სეტებად. stratify პარამეტრით ვინარჩუნებთ fraud-ის 3.5% თანაფარდობას ორივე ნაწილში.

In [5]:
from sklearn.model_selection import train_test_split

y = train["isFraud"]
X = train.drop(columns=["isFraud", "TransactionID"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

del train
gc.collect()

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

X_train: (472432, 432), X_val: (118108, 432)


# Cleaning
ვშლით 90%-ზე მეტი null-ის მქონე სვეტებს. დანარჩენ null-ებისთვის რიცხვითებს ვავსებთ -999-ით, ხოლო კატეგორიილებს ტექსტით "missing"

In [6]:
from sklearn.preprocessing import LabelEncoder

missing_pct = X_train.isnull().sum() / len(X_train)
high_null_cols = missing_pct[missing_pct > 0.9].index.tolist()
X_train = X_train.drop(columns=high_null_cols)
X_val = X_val.drop(columns=high_null_cols)
print(f"Dropped {len(high_null_cols)} high-null columns")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

X_train[num_cols] = X_train[num_cols].fillna(-999)
X_val[num_cols] = X_val[num_cols].fillna(-999)
X_train[cat_cols] = X_train[cat_cols].fillna("missing")
X_val[cat_cols] = X_val[cat_cols].fillna("missing")

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_val[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))
    label_encoders[col] = le

print(f"Final shape: {X_train.shape}, NaN: {X_train.isnull().sum().sum()}")

Dropped 12 high-null columns
Final shape: (472432, 420), NaN: 0


# Feature Engineering
უკვე არსებული ცვლადებიდან გამოგვყავს 7 ახალი ცვლადი, მაგალითად ტრანზაქციის საათი, მისი ათობითი ნაწილი, ბარათზე ტრანზაქციების საშუალო და ა.შ.

In [7]:
for df in [X_train, X_val]:
    df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
    df["Transaction_dow"] = (df["TransactionDT"] / 86400) % 7
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
    df["TransactionAmt_decimal"] = (df["TransactionAmt"] - df["TransactionAmt"].astype(int)).round(2)
    df["Card1_count"] = df["card1"].map(df["card1"].value_counts())
    df["Card1_TransactionAmt_mean"] = df["card1"].map(df.groupby("card1")["TransactionAmt"].mean())
    df["Amt_div_card1mean"] = df["TransactionAmt"] / (df["Card1_TransactionAmt_mean"] + 1)

print(f"Added 7 features. Shape: {X_train.shape}")

Added 7 features. Shape: (472432, 427)


/tmp/ipykernel_57/2475059092.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Transaction_hour"] = (df["TransactionDT"] / 3600) % 24
/tmp/ipykernel_57/2475059092.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Transaction_dow"] = (df["TransactionDT"] / 86400) % 7
/tmp/ipykernel_57/2475059092.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To 

# Feature Selection
შევიყვანოთ მონაცემები მცირე XGBOOST მოდელში, რათა გავიგოთ, რომელი ფიჩერებია მნიშვნელოვანი, და დავტოვოთ მხოლოდ ისენი. 

In [8]:
from xgboost import XGBClassifier

quick_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="auc",
    tree_method="hist",
    device="cuda",
    n_jobs=-1,
)
quick_model.fit(X_train, y_train)

importances = pd.Series(quick_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top_n = 100
selected_features = importances.head(top_n).index.tolist()

print(f"Selected top {top_n} features from {X_train.shape[1]}")
print(f"Top 15:")
print(importances.head(15))

X_train_sel = X_train[selected_features]
X_val_sel = X_val[selected_features]

Selected top 100 features from 427
Top 15:
V258     0.185711
V295     0.065253
V91      0.037192
V201     0.037109
V70      0.033160
V149     0.031223
V147     0.022042
C8       0.017921
V29      0.017145
id_17    0.016983
V294     0.016886
C14      0.016093
V308     0.015649
C4       0.015308
C12      0.014715
dtype: float32


# Training
ვწვრთნით Random Forest-ს სხვადასხვა კონფიგურაციით. 

In [9]:
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

def train_and_log_rf(run_name, params, X_tr, X_va, y_tr, y_va):
    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)

    train_pred = model.predict_proba(X_tr)[:, 1]
    val_pred = model.predict_proba(X_va)[:, 1]

    train_auc = roc_auc_score(y_tr, train_pred)
    val_auc = roc_auc_score(y_va, val_pred)
    gap = train_auc - val_auc

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_param("n_features", X_tr.shape[1])
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("overfit_gap", gap)
        mlflow.sklearn.log_model(model, name="model")

    print(f"{run_name:40s}  train_auc={train_auc:.4f}  val_auc={val_auc:.4f}  gap={gap:+.4f}")
    return model, val_auc

## Baseline
სტანდარტული Random Forest 100 ხით.

In [10]:
params_baseline = {
    "n_estimators": 100,
    "max_depth": 10,
    "min_samples_leaf": 5,
}

model_baseline, auc_baseline = train_and_log_rf(
    "RF_baseline", params_baseline,
    X_train_sel, X_val_sel, y_train, y_val,
)

2026/05/02 15:13:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_baseline at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/9a2db252436a44b29451d04b16dbfd67
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
RF_baseline                               train_auc=0.8802  val_auc=0.8727  gap=+0.0075


## Overfit ტესტი
ხეების სიღრმე გავხადოთ შეუზღუდავი overfitting-ის ტესტირებისთვის.

In [11]:
params_overfit = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_leaf": 1,
}

model_overfit, auc_overfit = train_and_log_rf(
    "RF_overfit_test", params_overfit,
    X_train_sel, X_val_sel, y_train, y_val,
)

2026/05/02 15:15:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_overfit_test at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/24c053e1502246658b5d88b283d5ab94
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
RF_overfit_test                           train_auc=1.0000  val_auc=0.9218  gap=+0.0782


## Tuned + Class Balancing
class_weight="balanced" გვეხმარება იშვიათ კლასზე კონცენტრირებაში

In [12]:
params_tuned = {
    "n_estimators": 200,
    "max_depth": 15,
    "min_samples_leaf": 3,
    "max_features": "sqrt",
    "class_weight": "balanced",
}

model_tuned, auc_tuned = train_and_log_rf(
    "RF_tuned", params_tuned,
    X_train_sel, X_val_sel, y_train, y_val,
)

2026/05/02 15:17:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_tuned at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/edafd8c8fe99466ab28b9d5639ad92dd
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
RF_tuned                                  train_auc=0.9547  val_auc=0.9156  gap=+0.0391


# შედეგების შედარება

In [13]:
results = {
    "baseline (depth=10)": auc_baseline,
    "overfit (depth=None)": auc_overfit,
    "tuned (depth=15, balanced)": auc_tuned,
}

print("=== RF Results ===")
for name, auc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:40s}  val_auc={auc:.4f}")

best_name = max(results, key=results.get)
print(f"\nBest: {best_name} (AUC={results[best_name]:.4f})")

=== RF Results ===
  overfit (depth=None)                      val_auc=0.9218
  tuned (depth=15, balanced)                val_auc=0.9156
  baseline (depth=10)                       val_auc=0.8727

Best: overfit (depth=None) (AUC=0.9218)


# საუკეთესო მოდელის არჩევა

In [14]:
all_models = {
    "baseline":  (model_baseline, auc_baseline),
    "overfit":   (model_overfit,  auc_overfit),
    "tuned":     (model_tuned,    auc_tuned),
}

best_name = max(all_models, key=lambda k: all_models[k][1])
best_model, best_auc = all_models[best_name]

print(f"Best run: {best_name} (val AUC = {best_auc:.4f})")

with mlflow.start_run(run_name="FINAL_RF_best") as run:
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("source_run", best_name)
    mlflow.log_metric("val_auc", best_auc)
    mlflow.sklearn.log_model(best_model, name="model")
    print("Logged FINAL RF model")

Best run: overfit (val AUC = 0.9218)


2026/05/02 15:19:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged FINAL RF model
🏃 View run FINAL_RF_best at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/a78d36167ca44762814d3546af250824
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


# Pipeline

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import gc

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_threshold=0.9):
        self.null_threshold = null_threshold
    def fit(self, X, y=None):
        X = X.copy()
        X.columns = [c.replace("id-", "id_") for c in X.columns]
        if "TransactionID" in X.columns:
            X = X.drop(columns=["TransactionID"])
        missing_pct = X.isnull().sum() / len(X)
        self.high_null_cols_ = missing_pct[missing_pct > self.null_threshold].index.tolist()
        X = X.drop(columns=self.high_null_cols_)
        self.num_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        X[self.num_cols_] = X[self.num_cols_].fillna(-999)
        X[self.cat_cols_] = X[self.cat_cols_].fillna("missing")
        self.label_encoders_ = {}
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(X[col].astype(str))
            self.label_encoders_[col] = le
        self.card1_counts_ = X["card1"].value_counts().to_dict()
        self.card1_amt_mean_ = X.groupby("card1")["TransactionAmt"].mean().to_dict()
        for col in self.cat_cols_:
            X[col] = self.label_encoders_[col].transform(X[col].astype(str))
        X = self._add_features(X)
        self.feature_columns_ = X.columns.tolist()
        return self
    def transform(self, X):
        X = X.copy()
        X.columns = [c.replace("id-", "id_") for c in X.columns]
        if "TransactionID" in X.columns:
            X = X.drop(columns=["TransactionID"])
        cols_to_drop = [c for c in self.high_null_cols_ if c in X.columns]
        X = X.drop(columns=cols_to_drop)
        for col in self.num_cols_:
            if col in X.columns: X[col] = X[col].fillna(-999)
        for col in self.cat_cols_:
            if col in X.columns: X[col] = X[col].fillna("missing")
        for col in self.cat_cols_:
            if col in X.columns:
                le = self.label_encoders_[col]
                known = set(le.classes_); fallback = le.classes_[0]
                X[col] = X[col].astype(str).apply(lambda v: v if v in known else fallback)
                X[col] = le.transform(X[col])
        X = self._add_features(X)
        for col in self.feature_columns_:
            if col not in X.columns: X[col] = 0
        return X[self.feature_columns_]
    def _add_features(self, X):
        X["Transaction_hour"] = (X["TransactionDT"] / 3600) % 24
        X["Transaction_dow"] = (X["TransactionDT"] / 86400) % 7
        X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        X["TransactionAmt_decimal"] = (X["TransactionAmt"] - X["TransactionAmt"].astype(int)).round(2)
        X["Card1_count"] = X["card1"].map(self.card1_counts_).fillna(0)
        X["Card1_TransactionAmt_mean"] = X["card1"].map(self.card1_amt_mean_).fillna(X["TransactionAmt"].mean())
        X["Amt_div_card1mean"] = X["TransactionAmt"] / (X["Card1_TransactionAmt_mean"] + 1)
        return X

DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
train_t = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_i = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
train_full = train_t.merge(train_i, on="TransactionID", how="left")
del train_t, train_i
gc.collect()

y_full = train_full["isFraud"]
X_full = train_full.drop(columns=["isFraud"])
del train_full
gc.collect()

X_tr_raw, X_va_raw, y_tr, y_va = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full,
)
del X_full
gc.collect()

rf_pipeline = Pipeline([
    ("preprocessor", FraudPreprocessor(null_threshold=0.9)),
    ("model", RandomForestClassifier(
        n_estimators=100, max_depth=None, min_samples_leaf=1,
        random_state=42, n_jobs=-1,
    )),
])

print("Fitting RF Pipeline on raw data...")
rf_pipeline.fit(X_tr_raw, y_tr)
val_pred = rf_pipeline.predict_proba(X_va_raw)[:, 1]
val_auc = roc_auc_score(y_va, val_pred)
print(f"RF Pipeline Val AUC: {val_auc:.4f}")

with mlflow.start_run(run_name="RF_Pipeline_FINAL") as run:
    mlflow.log_param("model", "RandomForest_Pipeline")
    mlflow.log_metric("val_auc", val_auc)
    mlflow.sklearn.log_model(rf_pipeline, name="model")
    print("RF Pipeline logged")

Fitting RF Pipeline on raw data...


/tmp/ipykernel_57/2772052204.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["Transaction_hour"] = (X["TransactionDT"] / 3600) % 24
/tmp/ipykernel_57/2772052204.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["Transaction_dow"] = (X["TransactionDT"] / 86400) % 7
/tmp/ipykernel_57/2772052204.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To g

RF Pipeline Val AUC: 0.9329


2026/05/02 15:48:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF Pipeline logged
🏃 View run RF_Pipeline_FINAL at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/c812b977dbec4188b41a4468d245486a
🧪 View experiment at: https://dagshub.com/llikl23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
